##### Copyright 2025 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemma 4 使用 Hugging Face Transformers 進行多標記預測 (MTP)

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/gemma/docs/mtp/mtp"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
</td> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/docs/mtp/mtp.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemma/cookbook/blob/main/docs/mtp/mtp.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemma%2Fcookbook%2Fmain%2Fdocs%2Fmtp%2Fmtp.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemma/cookbook/blob/main/docs/mtp/mtp.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td>
</table>

為了提高 Gemma 4 型號的 inference 速度，一系列新的自回歸「drafter」型號已與主系列一起發布。草稿模型不是僅依賴主要的 Gemma 4 個模型（稱為「目標」模型），而是在目標模型僅處理一個模型的時間內自回歸預測多個 tokens。該技術也稱為推測解碼。
在繪圖者預測了多個草稿tokens之後，目標模型現在只需驗證那些建議的草稿tokens。驗證是並行完成的，從而大大加快inference。它減少了目標模型必須為每個 token 執行的前向傳遞次數。因為我們的起草者產生了一個 tokens 序列用於驗證，所以我們將其稱為多 token 預測 (MTP) 頭。
![png](https://ai.google.dev/static/gemma/docs/mtp/image_files/mtp_overview.png)

為 Gemma 4 系列發布的草稿模型很小，並引入了多項增強功能來提高草稿 tokens 的品質並進一步加快 inference 的速度，例如使用目標模型啟動和 KV 快取來獲得更好的預測。
這些增強功能可顯著提高解碼速度，同時確保類似的質量，使這些 checkpoint 非常適合低延遲和設備上應用。

## 安裝 Python 軟體包

安裝執行Gemma 4 和Gemma 4 助理模型所需的Hugging Face 庫。

In [ ]:
# Install PyTorch & other libraries
!pip install torch accelerate

# Install the transformers library
!pip install transformers

## 載入模型

對於每個目標模型（Gemma 4模型中的主要模型之一），都有一個助手來幫助加速inference。因此，您將載入兩個模型：
- **目標**（例如`google/gemma-4-E2B-it`）：完整的Gemma 4目標模型
- **起草者**（例如`google/gemma-4-E2B-it-assistant`）：提出候選tokens的輕量級4層 MTP 起草者

請注意，*起草者*通常被稱為*助理*，因為該模型幫助較大的模型選擇要預測的tokens。
使用`transformers` 庫透過`AutoProcessor` 和`AutoModelForCausalLM` 類別建立`processor` 和`model` 的實例，如下列程式碼範例所示：

In [ ]:
TARGET_MODEL_ID = "google/gemma-4-E2B-it" # @param ["google/gemma-4-E2B-it", "google/gemma-4-E4B-it", "google/gemma-4-12B-it", "google/gemma-4-31B-it", "google/gemma-4-26B-A4B-it"]
ASSISTANT_MODEL_ID = TARGET_MODEL_ID + "-assistant"

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

# Target Model
processor = AutoProcessor.from_pretrained(TARGET_MODEL_ID)
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Assistant Model (the drafter)
assistant_model = AutoModelForCausalLM.from_pretrained(
    ASSISTANT_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/50 [00:00<?, ?it/s]

## Gemma 4 與助理

幸運的是，在 `transformers` 中使用助手非常簡單，需要您將助手模型傳遞給 `model.generate` 函數：

In [ ]:
# Process inputs with the `target_model`
messages = [
    {
        "role": "user",
        "content": "Explain the concepts of speculative decoding and MTP in 3 sentences."
    }
]
input_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=input_text, return_tensors="pt").to(target_model.device)

# `assistant_model=assistant_model` is all you need to enable MTP!
outputs = target_model.generate(
    **inputs,
    assistant_model=assistant_model,
    max_new_tokens=256,
    do_sample=False,
)

# Decode the response into text
response = processor.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

**Speculative decoding** is a technique where a smaller, faster language model (the "draft model") generates several candidate tokens, which are then quickly verified by a larger, more accurate model to produce a final, high-quality output much faster than decoding the large model alone. **MTP (Multi-Task Prediction)** involves training a single model to perform multiple related tasks simultaneously, allowing it to leverage shared knowledge across different objectives. Together, these methods aim to significantly accelerate the inference speed of large language models while maintaining or improving output quality.


在幕後，過程如下：
* 起草者建議 N tokens 自回歸生成
* 目標模型在 **一次** 前向傳遞中驗證所有 N tokens
* 起草的 tokens 有很高的機率被接受
* 起草的 tokens 機率較低被拒絕
* 由於目標模型進行前向傳遞，因此無論有多少起草的 tokens 被接受或拒絕，它總是會自行生成 1 token

## 草稿代幣

起草者可以產生任意數量的tokens以供目標模型驗證。但是，目標模型仍然可以選擇拒絕某些tokens。當它發生時，之後的所有 tokens 都將被忽略。
![png](https://ai.google.dev/static/gemma/docs/mtp/image_files/accept_reject.png)

因此，在使用不同的起草 tokens 數量值時，了解權衡非常重要。

**更多草稿tokens**
當您起草許多tokens（例如15）時，很有可能並非所有tokens都會被接受。因此，浪費計算的可能性更大。相比之下，當接受率較高時，它確實有加速inference的趨勢。
![png](https://ai.google.dev/static/gemma/docs/mtp/image_files/many_tokens.png)

**草稿較少tokens**
當您起草較少的 tokens 時，接受率往往會更高，因為 tokens 位置更接近初始 prompt 更準確。但是，由於僅起草了幾個tokens，從更快的起草器模型中獲得的速度會降低。
![png](https://ai.google.dev/static/gemma/docs/mtp/image_files/few_tokens.png)


幸運的是，您不必在`transformers`中試驗適合您用例的最佳值，因為您可以將`num_assistant_tokens_schedule`設定為“啟發式”，這將自動調整runtime處起草的tokens的數量：
* **所有 tokens 均已接受** -- 將要起草的 tokens 數量增加 2，因為起草者對於 prompt 相當準確。如果 tokens 也被接受，那麼增加起草的 tokens 數量可能會導致速度加快。
* **任何 tokens 被拒絕** -- 如果任何 tokens 被拒絕，則將起草的 tokens 數量減少 1。減少 tokens 的數量使得如果目標模型繼續拒絕大部分 tokens 則不會浪費太多起草的數量。

同樣，您可以透過在起草器中更新`num_assistant_tokens`來更新草稿tokens的編號，如下所示：

In [ ]:
# Update how many draft tokens are generated at the start of inference
assistant_model.generation_config.num_assistant_tokens = 4

# Update how the number of draft tokens are updated ("heuristic" for a dynamic schedule and "constant" for a constant schedule)
assistant_model.generation_config.num_assistant_tokens_schedule = "heuristic"

# Run with MTP
outputs = target_model.generate(
    **inputs,
    assistant_model=assistant_model,
    max_new_tokens=256,
    do_sample=False,
)

# Decode the response into text
response = processor.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

**Speculative decoding** is a technique where a smaller, faster language model (the "draft model") generates several candidate tokens, which are then verified by a larger, more accurate model to quickly produce a high-quality output. **MTP (Multi-Task Prediction)** involves training a single model to perform multiple related tasks simultaneously, allowing it to leverage shared knowledge across different objectives. Together, these methods aim to significantly speed up the inference process of large language models while maintaining or improving output quality.
